# DomainNet evaluation

Loads pre-trained SiamNet checkpoints from `RUNS_DIR` (or `newcode/weights/` as fallback), downloads DomainNet on demand into `BASE_DIR/data/domainnet`, evaluates each model on:

* **Regime (a) overall** — one 1200-pair test set (600 positive + 600 negative) with the same training-time augmentation pipeline applied to both branches; produces `FPR_overall_pct` and `Recall_overall_pct`.
* **Regime (b) per-stratum** — one 1200-pair test set per geometric stratum (`R90, R180, R270, HFlip, VFlip, R45, R135, T020`), with only the stratum's geometric transform applied to the positive view (no photometric block); produces `Recall_<stratum>_pct`.

Both regimes share a single deterministic source pool of 200 images per domain (1200 in total). All metrics are evaluated at fixed threshold $\tau = 0.5$. The single artefact written is `RUNS_DIR/paper_table.csv`.

In [ ]:
# Install runtime dependencies (safe to re-run; torch is left untouched
# because it is pre-installed with the matching CUDA build on the cluster).
import subprocess, sys, importlib

def _ensure(pip_name, import_name=None, min_version=None):
    name = import_name or pip_name
    spec = pip_name + (f">={min_version}" if min_version else "")
    try:
        mod = importlib.import_module(name)
        if min_version:
            from packaging.version import parse
            if parse(getattr(mod, "__version__", "0")) < parse(min_version):
                raise ImportError("version too old")
        return
    except (ImportError, ModuleNotFoundError):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

# Installed only if missing or below required version.
_ensure("torchvision", "torchvision", min_version="0.17")  # v2.JPEG, v2.RandomAffine
_ensure("timm",         "timm",        min_version="0.9")
_ensure("torchmetrics", "torchmetrics")
_ensure("scikit-learn", "sklearn")
_ensure("pandas",       "pandas")
_ensure("matplotlib",   "matplotlib")
_ensure("tqdm",         "tqdm")
_ensure("Pillow",       "PIL")
_ensure("ipywidgets",   "ipywidgets")
_ensure("packaging",    "packaging")

print("All runtime dependencies are ready.")


In [ ]:
import torch
import timm
import os
import random
import numpy as np

IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = "."

RUNS_DIR    = os.path.join(BASE_DIR, "runs")
os.makedirs(RUNS_DIR, exist_ok=True)

def save_siamnet_weights(model, run_name: str) -> str:
    """Full SiamNet state_dict (encoder + head) under RUNS_DIR."""
    path = os.path.join(RUNS_DIR, f"{run_name}_state.pt")
    torch.save(model.state_dict(), path)
    return path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

g = torch.Generator()
g.manual_seed(42)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
!nvidia-smi

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Publication style: colorblind-safe Okabe-Ito palette, serif fonts, vector PDF.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]

mpl.rcParams.update({
    "font.family":         "serif",
    "font.size":           11,
    "axes.labelsize":      12,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "legend.fontsize":     10,
    "legend.frameon":      False,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "figure.dpi":          120,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "axes.grid":           True,
    "grid.alpha":          0.25,
    "grid.linestyle":      "--",
    "lines.linewidth":     1.8,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=PALETTE)


def save_figure(fig, name):
    """Save a figure as both PDF (vector, paper-ready) and PNG (preview)."""
    pdf_path = os.path.join(RUNS_DIR, f"{name}.pdf")
    png_path = os.path.join(RUNS_DIR, f"{name}.png")
    fig.savefig(pdf_path)
    fig.savefig(png_path)
    return pdf_path, png_path


In [ ]:
# Download DomainNet (~17.5 GB across 6 domains) if not already present.
# Mirrors the COCO download style: wget -c for resume, unzip -n for skip-existing.
import os
import subprocess

DN_ROOT = os.path.join(BASE_DIR, "data", "domainnet")
DN_BASE_URL = "http://csr.bu.edu/ftp/visda/2019/multi-source"
DN_DOMAINS = ["real", "painting", "clipart", "quickdraw", "infograph", "sketch"]


def _have_domain(domain: str) -> bool:
    """True if domain dir exists and contains at least one image at any depth."""
    domain_dir = os.path.join(DN_ROOT, domain)
    if not os.path.isdir(domain_dir):
        return False
    for r, _, files in os.walk(domain_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                return True
    return False


os.makedirs(DN_ROOT, exist_ok=True)
for domain in DN_DOMAINS:
    if _have_domain(domain):
        print(f"  [skip ] {domain}: already present")
        continue
    url = f"{DN_BASE_URL}/{domain}.zip"
    zip_path = os.path.join(DN_ROOT, f"{domain}.zip")
    if not os.path.isfile(zip_path):
        print(f"  [get  ] {domain}: downloading {url}")
        subprocess.check_call(["wget", "-c", "--no-verbose", "-O", zip_path, url])
    print(f"  [unzip] {domain}: -> {DN_ROOT}")
    subprocess.check_call(["unzip", "-q", "-n", zip_path, "-d", DN_ROOT])

# Final report (recursive: counts images at any depth).
total = 0
for d in DN_DOMAINS:
    domain_dir = os.path.join(DN_ROOT, d)
    n = 0
    if os.path.isdir(domain_dir):
        for r, _, files in os.walk(domain_dir):
            n += sum(1 for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
    total += n
    print(f"  {d:11s}: {n} images")
print(f"DomainNet ready at {DN_ROOT} ({total} images total)")

# Free disk: drop zips after successful extraction.
for d in DN_DOMAINS:
    z = os.path.join(DN_ROOT, f"{d}.zip")
    if os.path.isfile(z) and _have_domain(d):
        os.remove(z)


In [ ]:
# Diagnose painting/clipart and force re-download if their image count is 0.
# Walks the domain dir recursively (handles deeper structures than 2 levels).
import os, glob, subprocess, shutil

DN_ROOT = os.path.join(BASE_DIR, "data", "domainnet")
DN_BASE_URL = "http://csr.bu.edu/ftp/visda/2019/multi-source"
SUSPECT = ["painting", "clipart", "real", "quickdraw", "infograph", "sketch"]


def _count_images(root):
    """Recursive image count (any depth)."""
    n = 0
    for r, _, files in os.walk(root):
        n += sum(1 for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
    return n


def _show_layout(root, depth=3):
    """Print the first few entries at each depth, so we can see the actual structure."""
    print(f"  layout under {root}:")
    if not os.path.isdir(root):
        print("    (does not exist)"); return
    for r, dirs, files in os.walk(root):
        rel = os.path.relpath(r, root)
        d = 0 if rel == "." else rel.count(os.sep) + 1
        if d > depth: continue
        sample = (dirs[:3] + files[:3])[:5]
        print(f"    [d={d}] {rel}: {len(dirs)} dirs, {len(files)} files; sample: {sample}")


def _force_redownload(domain):
    """Wipe the domain folder and any stale zip; download fresh; unzip."""
    domain_dir = os.path.join(DN_ROOT, domain)
    if os.path.isdir(domain_dir):
        print(f"  [wipe ] {domain_dir}")
        shutil.rmtree(domain_dir)
    zip_path = os.path.join(DN_ROOT, f"{domain}.zip")
    if os.path.isfile(zip_path):
        os.remove(zip_path)
    url = f"{DN_BASE_URL}/{domain}.zip"
    print(f"  [get  ] {url}")
    subprocess.check_call(["wget", "--no-verbose", "-O", zip_path, url])
    # Verify zip integrity before unzipping.
    rc = subprocess.call(["unzip", "-tq", zip_path])
    if rc != 0:
        print(f"  [err  ] {domain}.zip is corrupt; redownload failed")
        return 0
    print(f"  [unzip] {zip_path}")
    subprocess.check_call(["unzip", "-q", zip_path, "-d", DN_ROOT])
    os.remove(zip_path)
    return _count_images(domain_dir)


# Step 1: report current state recursively for ALL domains
for d in SUSPECT:
    domain_dir = os.path.join(DN_ROOT, d)
    n = _count_images(domain_dir)
    print(f"{d:11s}: {n} images (recursive)")
    if n == 0:
        _show_layout(domain_dir)

# Step 2: re-download empty domains
print()
for d in SUSPECT:
    domain_dir = os.path.join(DN_ROOT, d)
    if _count_images(domain_dir) == 0:
        print(f"=== re-downloading {d} ===")
        n = _force_redownload(d)
        print(f"  -> {n} images")


## Models

In [ ]:
import torchvision.transforms as transforms
from torchvision.transforms import v2, functional
import torch
import numpy as np
import random
import string
import io
from PIL import Image, ImageDraw, ImageFont


class RandomCompose(object):
    """Apply each transform with probability p (independently). Works on PIL or Tensor."""
    def __init__(self, transforms, p=0.3):
        self.p = p
        self.transforms = transforms

    def __call__(self, x):
        for t in self.transforms:
            if random.random() < self.p:
                x = t(x)
        return x


# === D4: tensor-native rotations 90/180/270 ===

class RandomRotation90Tensor(object):
    def __call__(self, x):
        k = random.randint(1, 3)
        return torch.rot90(x, k, dims=(-2, -1))


# === Gaussian noise (works on uint8 or float tensors) ===

class GaussianNoiseTensor(object):
    def __init__(self, std_range=(0.01, 0.05)):
        self.std_range = std_range

    def __call__(self, x):
        std = random.uniform(*self.std_range)
        if x.dtype == torch.uint8:
            std *= 255.0
            noise = torch.randn_like(x, dtype=torch.float32) * std
            return torch.clamp(x.float() + noise, 0, 255).to(torch.uint8)
        return torch.clamp(x + torch.randn_like(x) * std, 0, 1)


# === Watermark (kept PIL-based; text rendering is awkward on tensors) ===

class RandomWatermark(object):
    def __init__(self, alpha_range=(0.1, 0.4)):
        self.alpha_range = alpha_range

    def __call__(self, img):
        overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
        draw = ImageDraw.Draw(overlay)
        text = ''.join(random.choices(string.ascii_uppercase, k=random.randint(2, 5)))
        font_size = max(8, img.size[0] // 3)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except (OSError, IOError):
            font = ImageFont.load_default()
        x = random.randint(0, max(0, img.size[0] - font_size))
        y = random.randint(0, max(0, img.size[1] - font_size))
        alpha = int(255 * random.uniform(*self.alpha_range))
        color = (
            random.randint(100, 255),
            random.randint(100, 255),
            random.randint(100, 255),
            alpha
        )
        draw.text((x, y), text, fill=color, font=font)
        return Image.alpha_composite(img.convert('RGBA'), overlay).convert('RGB')


# Какую группу геометрических аугментаций добавлять поверх блока P в augmented loader.
#   'none' -- только photometric (для эквивариантных энкодеров).
#   'd4'   -- photometric + R90/180/270, HFlip, VFlip.
#   'se2'  -- photometric + непрерывный поворот в [0, 360) и непрерывный сдвиг.
GEOMETRIC_GROUP = 'se2'


def _identity_pipeline():
    """PIL -> uint8 tensor -> 32x32 -> float32."""
    return v2.Compose([
        v2.PILToTensor(),
        v2.Resize((32, 32), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
    ])


def get_augmentations(group=None, aug=True):
    """Build augmentation pipeline. Returns float32 (C,H,W) tensors of size 32x32."""
    if not aug:
        return _identity_pipeline()

    if group is None:
        group = GEOMETRIC_GROUP

    # PIL stage: watermark draws text via ImageDraw (no tensor equivalent).
    pil_ops = [RandomWatermark(alpha_range=(0.1, 0.4))]

    # Tensor stage (operates on uint8 CHW tensors after PILToTensor).
    photometric_t = [
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        GaussianNoiseTensor(std_range=(0.01, 0.05)),
        v2.JPEG(quality=(10, 50)),
    ]
    d4_block = [
        RandomRotation90Tensor(),
        v2.RandomHorizontalFlip(p=1.0),
        v2.RandomVerticalFlip(p=1.0),
    ]
    se2_block = [
        v2.RandomRotation(degrees=(0, 360), expand=False),
        v2.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    ]

    if group == 'none':
        tensor_ops = photometric_t
    elif group == 'd4':
        tensor_ops = photometric_t + d4_block
    elif group == 'se2':
        tensor_ops = photometric_t + se2_block
    else:
        raise ValueError(f"unknown group {group!r}; expected one of 'none', 'd4', 'se2'")

    return v2.Compose([
        RandomCompose(pil_ops, p=0.3),
        v2.PILToTensor(),
        RandomCompose(tensor_ops, p=0.3),
        v2.Resize((32, 32), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
    ])


# === PIL-based companions used by DomainNet per-class evaluation ===
# Eval pipeline operates on PIL images one at a time, so these are kept as
# simple PIL implementations even though training uses the v2 tensor versions
# for speed.

class GaussianNoise(object):
    def __init__(self, std_range=(0.01, 0.05)):
        self.std_range = std_range
    def __call__(self, img):
        img_np = np.array(img).astype(np.float32) / 255.0
        std = random.uniform(*self.std_range)
        noise = np.random.normal(0, std, img_np.shape).astype(np.float32)
        img_np = np.clip(img_np + noise, 0, 1)
        return Image.fromarray((img_np * 255).astype(np.uint8))


class JPEGCompression(object):
    def __init__(self, quality_range=(10, 50)):
        self.quality_range = quality_range
    def __call__(self, img):
        quality = random.randint(*self.quality_range)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert('RGB')


class RandomTranslate(object):
    def __init__(self, max_fraction=0.2, fill=0):
        self.max_fraction = max_fraction
        self.fill = fill
    def __call__(self, img):
        w, h = img.size
        dx = int(round(random.uniform(-self.max_fraction, self.max_fraction) * w))
        dy = int(round(random.uniform(-self.max_fraction, self.max_fraction) * h))
        return functional.affine(
            img, angle=0.0, translate=(dx, dy), scale=1.0, shear=(0.0, 0.0),
            interpolation=transforms.InterpolationMode.BILINEAR, fill=self.fill,
        )


In [ ]:
from timm.models.vision_transformer import VisionTransformer
import torch.nn as nn

class ViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = VisionTransformer(
            img_size=32,           # Стандартный размер входного изображения для предобученных весов
            patch_size=4,          # Размер патча (ViT-.../16)
            in_chans=3,
            num_classes = 0,
            embed_dim=192,          # Ключевой параметр: скрытая размерность для Tiny
            depth=12,               # Количество слоёв трансформера
            num_heads=3,            # Количество голов внимания (192 / 64 = 3)
            mlp_ratio=4.0,          # Соотношение размера MLP к embed_dim (192 * 4 = 768)
            class_token=True,       # Использовать классификационный токен
            global_pool='token',    # Использовать class token для классификации
        )
        self.feature_dim = 192

    def forward(self, x):
        return self.backbone(x)


In [ ]:
!git clone -q https://github.com/davnords/octic-vits.git

import sys
sys.path.insert(0, "octic-vits")

In [ ]:
from octic_vits import OcticVisionTransformer
import os
import torch.nn as nn

class OcticViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_dim = 192

        self.backbone = OcticVisionTransformer(
            img_size=32,
            patch_size=4,
            embed_dim=192,
            depth=12,
            num_heads=3,
            invariant=True,
            num_classes=0,
        )

    def forward(self, x):
        out = self.backbone(x)
        return out


In [ ]:
import sys
import os
import torch.nn as nn

_harm_paths = [
    os.path.join(BASE_DIR, "newcode"),
    BASE_DIR,
    os.path.join(os.getcwd(), "newcode"),
    os.getcwd(),
]
for _p in _harm_paths:
    if _p and os.path.isfile(os.path.join(_p, "harmformer_encoder.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

from harmformer_encoder import HarmformerEncoder


class HarmformerViTEncoder(nn.Module):
    """Tiny Harmformer; same embedding dim as ViT/Octic (192)."""

    def __init__(self):
        super().__init__()
        self.backbone = HarmformerEncoder(
            img_size=32,
            stem_channels=[16, 32],
            encoder_dim=64,       # feature_dim = 3 * 64 = 192
            encoder_depth=4,
            num_heads=4,
        )
        self.feature_dim = self.backbone.feature_dim

    def forward(self, x):
        return self.backbone(x)


In [ ]:
class HeadModel(nn.Module):
    """Advanced head that accepts pre-computed difference AND product embeddings.
    
    This architecture is designed to work with pre-computed difference vectors AND element-wise products.
    The inputs should already be in the format:
    - diff_embeddings: [|emb1 - emb2|] for all pairs
    - product_embeddings: [emb1 * emb2] for all pairs
    """
    
    def __init__(self, feature_dim, dropout_rate=0.1):
        """
        Args:
            feature_dim (int): Dimension of input feature vectors
            dropout_rate (float): Dropout probability. Defaults to 0.1.
        """
        super().__init__()
        
        self.diff_pathway = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        self.product_pathway = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate)
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(2 * feature_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(feature_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, diff_embeddings, product_embeddings):
        """
        Forward pass with pre-computed difference and product features.
        
        Args:
            diff_embeddings: Tensor of shape [batch_size, feature_dim] 
                representing pre-computed |embedding1 - embedding2|
            product_embeddings: Tensor of shape [batch_size, feature_dim]
                representing pre-computed element-wise product of embeddings
                
        Returns:
            Tensor of shape [batch_size, 1] with similarity probabilities (after sigmoid)
        """
        diff_features = self.diff_pathway(diff_embeddings)
        product_features = self.product_pathway(product_embeddings)
        
        combined = torch.cat([diff_features, product_features], dim=1)
        return self.fusion(combined)


In [ ]:
class SiamNet(nn.Module):
    def __init__(self, encoder):
        super(SiamNet, self).__init__()
        self.encoder = encoder

        self.head = HeadModel(self.encoder.feature_dim)

    def forward(self, batch1: torch.Tensor, batch2: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the siamese network.

        Args:
            batch1: First batch of images [batch_size, channels, height, width]
            batch2: Second batch of images [batch_size, channels, height, width]

        Returns:
            torch.Tensor: Similarity matrix [batch_size1, batch_size2]
        """
        emb1 = self.encoder(batch1)
        emb2 = self.encoder(batch2)

        batch_size1, batch_size2 = emb1.size(0), emb2.size(0)
        emb1_expanded = emb1.unsqueeze(1).expand(-1, batch_size2, -1)
        emb2_expanded = emb2.unsqueeze(0).expand(batch_size1, -1, -1)

        # Pre-compute the difference and product vectors
        diff_embeddings = torch.abs(emb1_expanded - emb2_expanded)
        product_embeddings = emb1_expanded * emb2_expanded
        
        # Reshape for head model
        diff_embeddings = diff_embeddings.view(-1, self.encoder.feature_dim)
        product_embeddings = product_embeddings.view(-1, self.encoder.feature_dim)

        # Pass pre-computed features to the head
        logits = self.head(diff_embeddings, product_embeddings)
            
        logits = logits.view(batch_size1, batch_size2)
        return logits


## Dataset glue: pull `Dataset` / `DataLoader` and `Image` into scope

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image


## DomainNet pair builder

In [ ]:
import os
import random
from typing import Callable, Dict, List, Tuple

DOMAIN_NAMES = ["real", "painting", "clipart", "quickdraw", "infograph", "sketch"]


def _resolve_domainnet_root() -> str:
    candidates = [
        "/kaggle/input/domainnet/domainnet",
        "/kaggle/input/domainnet",
        os.path.join(BASE_DIR, "data", "domainnet"),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError("DomainNet root not found in: " + ", ".join(candidates))


def _list_domain_images(root: str, domain: str, max_per_domain: int = 4000) -> List[str]:
    """Recursively list images for a DomainNet domain. Robust to any depth."""
    domain_dir = os.path.join(root, domain)
    if not os.path.isdir(domain_dir):
        for entry in os.scandir(root):
            if entry.is_dir():
                cand = os.path.join(entry.path, domain)
                if os.path.isdir(cand):
                    domain_dir = cand
                    break
        else:
            raise RuntimeError(f"domain dir {domain!r} not found under {root}")
    files: List[str] = []
    for r, _, fs in os.walk(domain_dir):
        for f in fs:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                files.append(os.path.join(r, f))
    if not files:
        raise RuntimeError(f"no images found for domain {domain!r} under {domain_dir}")
    rng = random.Random(42)
    rng.shuffle(files)
    return files[:max_per_domain]


# Single deterministic source pool of 200 images per domain, shared across
# every stratum AND across the overall regime. 200 * 6 = 1200 source images,
# matching the paper's "Test set construction" paragraph.
SOURCE_POOL_SIZE = 200
_SOURCE_POOL_CACHE: Dict[str, List[str]] = {}


def _domain_source_pool(domain: str, n: int = SOURCE_POOL_SIZE,
                        seed: int = 42) -> List[str]:
    if domain in _SOURCE_POOL_CACHE:
        return _SOURCE_POOL_CACHE[domain]
    files = _list_domain_images(_resolve_domainnet_root(), domain)
    if len(files) < n:
        raise RuntimeError(f"need >= {n} images in {domain!r}, have {len(files)}")
    rng = random.Random(seed)
    rng.shuffle(files)
    pool = files[:n]
    _SOURCE_POOL_CACHE[domain] = pool
    return pool


# Geometric strata used in the paper table (Section 6, Table tab:per-stratum).
# The "Id" stratum is intentionally absent: identity gives 100% Recall by
# construction and carries no signal about geometric robustness.
def _make_strata_transforms() -> Dict[str, Callable]:
    return {
        "R90":   (lambda img: functional.rotate(img, 90, expand=True)),
        "R180":  (lambda img: functional.rotate(img, 180, expand=True)),
        "R270":  (lambda img: functional.rotate(img, 270, expand=True)),
        "HFlip": (lambda img: functional.hflip(img)),
        "VFlip": (lambda img: functional.vflip(img)),
        "R45":   (lambda img: functional.rotate(img, 45, expand=True)),
        "R135":  (lambda img: functional.rotate(img, 135, expand=True)),
        "T020":  RandomTranslate(max_fraction=0.2),
    }


class DomainNetPairs(Dataset):
    """Per-stratum balanced pair set for one (domain, stratum) cell.

    100 positive pairs and 100 negative pairs are drawn from the shared
    200-image source pool of the domain. Across 6 domains this gives 1200
    pairs per stratum (600 pos + 600 neg), matching the paper's per-stratum
    regime (Section 6, Test set construction).

    Positive pair: (anchor, transform_fn(anchor)) -- ONLY the stratum's
    geometric transformation is applied; the photometric block is OFF, so
    per-stratum Recall isolates the encoder's response to that single
    transformation.
    Negative pair: (img1, img2) -- two distinct sources from the same domain
    pool, no transformation applied.

    Positive sources are identical across strata so per-stratum Recall
    differences come purely from the geometric transformation, not from the
    underlying image content. Positive sources may reappear inside negative
    pairs (image reuse is allowed at the source level), but every (pair) is
    distinct.
    """

    def __init__(self, domain: str, stratum: str,
                 n_pos: int = 100, n_neg: int = 100, seed: int = 42):
        self.domain = domain
        self.stratum = stratum
        self.transform_fn = _make_strata_transforms()[stratum]

        pool = _domain_source_pool(domain)              # fixed 200/domain
        if n_pos > len(pool):
            raise RuntimeError(f"n_pos={n_pos} exceeds pool size {len(pool)}")
        if 2 * n_neg > len(pool):
            raise RuntimeError(
                f"n_neg={n_neg} requires {2 * n_neg} distinct sources, pool has {len(pool)}"
            )

        # Positives: first n_pos sources of the pool. Identical across strata.
        self.pos_sources = list(pool[:n_pos])

        # Negatives: shuffle the pool with a fixed seed and pair adjacent
        # entries; positive sources may reappear here, but every negative pair
        # consists of two distinct images.
        rng = random.Random(seed + 1)
        shuffled = list(pool)
        rng.shuffle(shuffled)
        self.neg_pairs = [
            (shuffled[2 * i], shuffled[2 * i + 1]) for i in range(n_neg)
        ]

        self.to_tensor = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.pos_sources) + len(self.neg_pairs)

    def __getitem__(self, idx):
        n_pos = len(self.pos_sources)
        if idx < n_pos:
            path = self.pos_sources[idx]
            img = Image.open(path).convert("RGB")
            view = self.transform_fn(img)
            return self.to_tensor(img), self.to_tensor(view), 1
        path1, path2 = self.neg_pairs[idx - n_pos]
        img1 = Image.open(path1).convert("RGB")
        img2 = Image.open(path2).convert("RGB")
        return self.to_tensor(img1), self.to_tensor(img2), 0


## Per-stratum and overall evaluation helpers (fixed threshold $\tau = 0.5$)

In [ ]:
import numpy as np
import pandas as pd

# Fixed decision threshold matching training time. Not re-tuned on the test set.
FIXED_THRESHOLD = 0.5


@torch.no_grad()
def _score_pairs(model, loader, device):
    model.eval()
    scores, labels = [], []
    for x1, x2, y in loader:
        x1, x2 = x1.to(device), x2.to(device)
        emb1 = model.encoder(x1)
        emb2 = model.encoder(x2)
        diff = torch.abs(emb1 - emb2)
        prod = emb1 * emb2
        s = model.head(diff, prod).view(-1).cpu().numpy()
        scores.append(s)
        labels.append(y.numpy())
    return np.concatenate(scores), np.concatenate(labels)


def _recall_fpr_at_threshold(scores, labels, thr=FIXED_THRESHOLD):
    """Return (recall, fpr) for binary predictions thresholded at thr."""
    pred = (scores >= thr).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    tn = int(((pred == 0) & (labels == 0)).sum())
    recall_v = tp / max(1, tp + fn)
    fpr_v    = fp / max(1, fp + tn)
    return recall_v, fpr_v


# ----- Per-stratum regime (b) -----------------------------------------------
# One separate 1200-pair test per stratum (600 pos + 600 neg). Positive pair
# applies only the stratum's geometric transform; photometric block is OFF.

def evaluate_per_stratum(model, strata, domains=DOMAIN_NAMES, batch_size=32):
    """Return {stratum: recall} at fixed threshold tau = FIXED_THRESHOLD."""
    out = {}
    for st in strata:
        ss, ys = [], []
        for dom in domains:
            ds = DomainNetPairs(dom, st)
            ld = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
            s, y = _score_pairs(model, ld, device)
            ss.append(s); ys.append(y)
        s = np.concatenate(ss); y = np.concatenate(ys)
        recall_v, _ = _recall_fpr_at_threshold(s, y)
        out[st] = recall_v
    return out


# ----- Overall regime (a) ---------------------------------------------------
# Single 1200-pair test (600 pos + 600 neg), built on top of the SAME 200/domain
# source pool used by the per-stratum regime. Both branches of every pair go
# through the training-time augmentation pipeline of `group` (Section sec:aug):
# photometric block always, plus the geometric block matched to the model's
# training group ('none' / 'd4' / 'se2'). This is the regime that the paper
# reports as global FPR and overall Recall.

def _build_overall_pairs(group, n_pos_per_domain=100, n_neg_per_domain=100,
                         seed=42):
    """Return (x1, x2, y) tensors for the 1200-pair overall test of one model.

    Positives: (identity-resized anchor, training-pipeline-augmented anchor)
    Negatives: two distinct sources of the same domain, BOTH passed through
               the same training-time augmentation pipeline.
    """
    aug      = get_augmentations(group=group, aug=True)
    identity = _identity_pipeline()

    x1_list, x2_list, y_list = [], [], []
    for dom in DOMAIN_NAMES:
        pool = _domain_source_pool(dom)
        # positives -- same anchors as the per-stratum positives.
        for path in pool[:n_pos_per_domain]:
            img = Image.open(path).convert("RGB")
            x1_list.append(identity(img))
            x2_list.append(aug(img))
            y_list.append(1)
        # negatives -- same shuffled pool as DomainNetPairs (seed+1).
        rng = random.Random(seed + 1)
        shuffled = list(pool)
        rng.shuffle(shuffled)
        for i in range(n_neg_per_domain):
            p1, p2 = shuffled[2 * i], shuffled[2 * i + 1]
            img1 = Image.open(p1).convert("RGB")
            img2 = Image.open(p2).convert("RGB")
            x1_list.append(aug(img1))
            x2_list.append(aug(img2))
            y_list.append(0)

    return (torch.stack(x1_list),
            torch.stack(x2_list),
            torch.tensor(y_list, dtype=torch.long))


@torch.no_grad()
def evaluate_overall(model, group, batch_size=32):
    """Return overall {recall, fpr, n_pos, n_neg} at fixed threshold."""
    x1, x2, y = _build_overall_pairs(group)
    model.eval()
    s_chunks = []
    for i in range(0, len(y), batch_size):
        b1 = x1[i : i + batch_size].to(device)
        b2 = x2[i : i + batch_size].to(device)
        emb1 = model.encoder(b1); emb2 = model.encoder(b2)
        diff = torch.abs(emb1 - emb2); prod = emb1 * emb2
        s = model.head(diff, prod).view(-1).cpu().numpy()
        s_chunks.append(s)
    s = np.concatenate(s_chunks)
    y_np = y.numpy()
    recall, fpr = _recall_fpr_at_threshold(s, y_np)
    return {"recall": recall, "fpr": fpr,
            "n_pos": int((y_np == 1).sum()),
            "n_neg": int((y_np == 0).sum())}


## Load checkpoints (RUNS_DIR or newcode/weights/ fallback) and run eval

In [ ]:
STRATA = [
    "R90", "R180", "R270", "HFlip", "VFlip", "R45", "R135", "T020",
]

# Architectures included in the paper table (Section 6, Table tab:per-stratum).
ENCODER_FACTORIES = {
    "vit_tiny_photometric":   ViTEncoder,
    "vit_tiny_d4_augs":       ViTEncoder,
    "octic_photometric":      OcticViTEncoder,
    "vit_tiny_se2_augs":      ViTEncoder,
    "harmformer_photometric": HarmformerViTEncoder,
}

# Training-time geometric augmentation group of each model. The overall regime
# replays this group for the same model so that the test distribution matches
# the training distribution (Section sec:aug).
MODEL_TRAINING_GROUP = {
    "vit_tiny_photometric":   "none",
    "vit_tiny_d4_augs":       "d4",
    "octic_photometric":      "none",
    "vit_tiny_se2_augs":      "se2",
    "harmformer_photometric": "none",
}

WEIGHT_DIRS = [
    RUNS_DIR,
    os.path.join(BASE_DIR, "newcode", "weights"),
    os.path.join(BASE_DIR, "weights"),
]


def _find_state_path(run_name):
    fname = f"{run_name}_state.pt"
    for d in WEIGHT_DIRS:
        p = os.path.join(d, fname)
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(
        f"checkpoint {fname} not found in any of: {', '.join(WEIGHT_DIRS)}"
    )


def _load_siamnet(run_name):
    encoder_cls = ENCODER_FACTORIES[run_name]
    model = SiamNet(encoder_cls()).to(device)
    state_path = _find_state_path(run_name)
    print(f"loading {run_name} from {state_path}")
    model.load_state_dict(torch.load(state_path, map_location=device))
    model.eval()
    return model


eval_results = {}
for run_name in ENCODER_FACTORIES:
    try:
        m = _load_siamnet(run_name)
    except FileNotFoundError as e:
        print(f"skip {run_name}: {e}")
        continue

    group = MODEL_TRAINING_GROUP[run_name]
    print(f"evaluating {run_name} (training group = {group!r})")

    # Regime (a) overall: 1200-pair test with training-time augmentation.
    overall = evaluate_overall(m, group)
    print(f"  overall : N={overall['n_pos'] + overall['n_neg']:4d} "
          f"(pos={overall['n_pos']}, neg={overall['n_neg']})  "
          f"Recall={overall['recall'] * 100:6.2f}%  "
          f"FPR={overall['fpr'] * 100:6.2f}%")

    # Regime (b) per-stratum: one 1200-pair test per geometric stratum.
    per_stratum = evaluate_per_stratum(m, STRATA)
    for st, r in per_stratum.items():
        print(f"  stratum {st:6s}: Recall={r * 100:6.2f}%")

    eval_results[run_name] = {
        "training_group":     group,
        "overall_recall":     overall["recall"],
        "overall_fpr":        overall["fpr"],
        "per_stratum_recall": per_stratum,
    }
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Build paper table

In [ ]:
# Build the paper table (Section 6, Table tab:per-stratum) directly from
# eval_results. Columns:
#   training_group, FPR_overall_pct, Recall_overall_pct        -- regime (a)
#   Recall_<stratum>_pct for every stratum in STRATA           -- regime (b)
# Saved to RUNS_DIR/paper_table.csv -- single artefact consumed by the LaTeX
# paper.

CSV_PATH = os.path.join(RUNS_DIR, "paper_table.csv")

if not eval_results:
    print("no checkpoints loaded; train the per-paper runs first")
else:
    rows = []
    for run_name, res in eval_results.items():
        row = {
            "model":              run_name,
            "training_group":     res["training_group"],
            "FPR_overall_pct":    float(res["overall_fpr"])    * 100.0,
            "Recall_overall_pct": float(res["overall_recall"]) * 100.0,
        }
        for st in STRATA:
            row[f"Recall_{st}_pct"] = float(res["per_stratum_recall"][st]) * 100.0
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model")
    df.to_csv(CSV_PATH)
    assert os.path.isfile(CSV_PATH), f"failed to write {CSV_PATH}"
    print(f"Saved paper table: {CSV_PATH} ({df.shape[0]} rows x {df.shape[1]} cols)")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)
    print(df.round(2).to_string())
